In [8]:
!pip install emoji
!pip install lingua-language-detector
!pip install python-dotenv

In [9]:
from googleapiclient.discovery import build
import pandas as pd
from lingua import Language, LanguageDetectorBuilder
import os
from dotenv import load_dotenv

In [12]:
load_dotenv('/content/api_key.env')
API_KEY = os.environ.get('api_key')
youtube = build('youtube', 'v3', developerKey=API_KEY)

video_id = 'tJbzMqJGH4k'

video_response = youtube.videos().list(
    part='snippet',
    id=video_id
).execute()
video_snippet = video_response['items'][0]['snippet']

detector = LanguageDetectorBuilder.from_all_languages().build()
def is_english(text):
  if len(text.split()) < 3:
    return False
  language = detector.detect_language_of(text)
  return language == Language.ENGLISH

# Fetch comments
print("Fetching Comments...")
comments = []
nextPageToken = None
while len(comments) < 30000:
  request = youtube.commentThreads().list(
      part='snippet',
      videoId=video_id,
      maxResults=100,
      pageToken=nextPageToken
  )
  response = request.execute()
  for item in response['items']:
      comment = item['snippet']['topLevelComment']['snippet']
      if is_english(comment['textDisplay']):
        if comment['authorChannelId']['value'] != video_snippet['channelId']:
          comments.append(comment['textDisplay'])
  nextPageToken = response.get('nextPageToken')

  if not nextPageToken:
      break

# Splitting the response for channelID
print("Video Title: " + video_snippet['title'])
print("Youtube Channel: " + video_snippet['channelTitle'])
print("Total Comments:", len(comments))
comments[:10]

Fetching Comments...
Video Title: Grand Theft Auto VI: An Extended Look
Youtube Channel: Rockstar Games
Total Comments: 30010


['14 days left let&#39;s go',
 'OMG this is so reali',
 'already been two months since this released BTW',
 'Best world game',
 'Only 2 weeks left friends...',
 '2 weeks left',
 'Looks the same',
 'So glad I booked 2 weeks off work for this 🎉🎉🎉',
 '13 years is a long time',
 'I watched this video like 1 million times .. Can&#39;t wait till November 19 🥹😍']

In [13]:
df_komen = pd.DataFrame(comments, columns=['comments'])
df_komen.to_csv('comments.csv', index=False)
df_komen.head(10)

,comments
0,14 days left let&#39;s go
1,OMG this is so reali
2,already been two months since this released BTW
3,Best world game
4,Only 2 weeks left friends...
5,2 weeks left
6,Looks the same
7,So glad I booked 2 weeks off work for this 🎉🎉🎉
8,13 years is a long time
9,I watched this video like 1 million times .. C...
